# Example 10.2.1 / 1: Find the Maximum Flow

Companion notebook for **10.2.1 Maximum Flow and Shortest Path** in the AMPL lab Documentation, section "Find the Maximum Flow". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Find the Maximum Flow

For intermediate nodes, inflow equals outflow. The source and sink are excluded from those conservation rows, and the objective maximizes flow leaving the source.

If arcs into the source or out of the sink are allowed, “flow leaving the source” can overstate
net throughput. Either prohibit those arcs as below, maximize net source outflow, or introduce a
single return arc from sink to source and maximize its flow. State which convention is used.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;  # network nodes
param source symbolic in NODES;  # where flow enters
param sink symbolic in NODES, != source;  # where flow leaves
# no arcs into source or out of sink
set ARCS within (NODES diff {sink}) cross (NODES diff {source});
param capacity {ARCS} >= 0;  # maximum flow on the arc

var Flow {(i, j) in ARCS} >= 0, <= capacity[i, j];  # flow from i to j

maximize Throughput:  # total flow leaving the source
    sum {(source, j) in ARCS} Flow[source, j];

subject to Conservation {k in NODES diff {source, sink}}:  # inflow equals outflow at inner nodes
    sum {(i, k) in ARCS} Flow[i, k]
  = sum {(k, j) in ARCS} Flow[k, j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NODES"] = ["s", "a", "b", "t"]
ampl.param["source"] = "s"
ampl.param["sink"] = "t"
ampl.set["ARCS"] = [("s", "a"), ("s", "b"), ("a", "b"), ("a", "t"), ("b", "t")]
ampl.param["capacity"] = {("s", "a"): 4, ("s", "b"): 3, ("a", "b"): 1, ("a", "t"): 2, ("b", "t"): 5}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.obj["Throughput"].value())

HiGHS 1.14.0: optimal solution; objective 6
0 simplex iterations
0 barrier iterations
solved
6.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
set NODES := s a b t;
param source := s;
param sink := t;
set ARCS := (s,a) (s,b) (a,b) (a,t) (b,t);
param capacity := [s,a] 4 [s,b] 3 [a,b] 1 [a,t] 2 [b,t] 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Throughput;

HiGHS 1.14.0: optimal solution; objective 6
0 simplex iterations
0 barrier iterations
Throughput = 6

